In [7]:
%pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import requests
import pandas as pd
from datetime import datetime
from io import StringIO

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []

start_year = datetime.now().year - 5
end_year = datetime.now()

session = requests.Session()

for year in range(start_year, end_year.year + 1):

    start_date = f"{year}-01-01T00:00:00"

    # Current year -> stop at current date/time
    if year == end_year.year:
        year_end_date = end_year.strftime("%Y-%m-%dT%H:%M:%S")
    else:
        # Include the complete December 31
        year_end_date = f"{year}-12-31T23:59:59"

    print(f"\nFetching {start_date} to {year_end_date}")

    offset = 1
    limit = 20000

    while True:

        common_params = {
            "starttime": start_date,
            "endtime": year_end_date,
            "limit": limit,
            "offset": offset,
            "orderby": "time-asc",
            "eventtype": "earthquake"
        }

        # ----------------------------------------
        # 1. Fetch GeoJSON data
        # ----------------------------------------

        geo_params = {
            **common_params,
            "format": "geojson"
        }

        response = session.get(
            url,
            params=geo_params,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        earthquakes = data["features"]

        if not earthquakes:
            break

        # ----------------------------------------
        # 2. Fetch CSV data for extra fields
        # ----------------------------------------

        csv_params = {
            **common_params,
            "format": "csv"
        }

        csv_response = session.get(
            url,
            params=csv_params,
            timeout=60
        )

        csv_response.raise_for_status()

        csv_df = pd.read_csv(
            StringIO(csv_response.text)
        )

        # Create dictionary indexed by earthquake id
        quality_fields = {}

        if not csv_df.empty:

            wanted_quality_columns = [
                "id",
                "magError",
                "depthError",
                "magNst",
                "locationSource",
                "magSource"
            ]

            quality_df = csv_df[
                wanted_quality_columns
            ].copy()

            quality_fields = (
                quality_df
                .set_index("id")
                .to_dict("index")
            )

        # ----------------------------------------
        # 3. Extract required fields
        # ----------------------------------------

        for earthquake in earthquakes:

            properties = earthquake["properties"]
            coordinates = earthquake["geometry"]["coordinates"]

            earthquake_id = earthquake["id"]

            # Extra CSV fields for same event
            quality = quality_fields.get(
                earthquake_id,
                {}
            )

            record = {

                # 1
                "id": earthquake_id,

                # 2
                "time": properties.get("time"),

                # 3
                "updated": properties.get("updated"),

                # 4
                "latitude": coordinates[1],

                # 5
                "longitude": coordinates[0],

                # 6
                "depth_km": coordinates[2],

                # 7
                "mag": properties.get("mag"),

                # 8
                "magType": properties.get("magType"),

                # 9
                "place": properties.get("place"),

                # 10
                "status": properties.get("status"),

                # 11
                "tsunami": properties.get("tsunami"),

                # 12
                "sig": properties.get("sig"),

                # 13
                "net": properties.get("net"),

                # 14
                "nst": properties.get("nst"),

                # 15
                "dmin": properties.get("dmin"),

                # 16
                "rms": properties.get("rms"),

                # 17
                "gap": properties.get("gap"),

                # 18
                "magError": quality.get("magError"),

                # 19
                "depthError": quality.get("depthError"),

                # 20
                "magNst": quality.get("magNst"),

                # 21
                "locationSource": quality.get("locationSource"),

                # 22
                "magSource": quality.get("magSource"),

                # 23
                "types": properties.get("types"),

                # 24
                "ids": properties.get("ids"),

                # 25
                "sources": properties.get("sources"),

                # 26
                "type": properties.get("type")
            }

            all_records.append(record)

        print(
            f"Fetched {len(earthquakes)} records "
            f"| Total: {len(all_records)}"
        )

        # Last page
        if len(earthquakes) < limit:
            break

        offset += limit


Fetching 2021-01-01T00:00:00 to 2021-12-31T23:59:59


KeyboardInterrupt: 

In [ ]:
all_records

[{'id': 'pr2021001027',
  'time': 1609459299400,
  'updated': 1611999858313,
  'latitude': 17.9951,
  'longitude': -67.0525,
  'depth_km': 5,
  'mag': 1.75,
  'magType': 'md',
  'place': '2 km NNW of La Parguera, Puerto Rico',
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 47,
  'net': 'pr',
  'nst': 4,
  'dmin': 0.027,
  'rms': 0.08,
  'gap': 120,
  'magError': 0.12,
  'depthError': 0.83,
  'magNst': 4.0,
  'locationSource': 'pr',
  'magSource': 'pr',
  'types': ',origin,phase-data,',
  'ids': ',pr2021001027,',
  'sources': ',pr,',
  'type': 'earthquake'},
 {'id': 'nc73505360',
  'time': 1609459450210,
  'updated': 1609797306182,
  'latitude': 38.0473333,
  'longitude': -118.7285,
  'depth_km': 5.25,
  'mag': 1.89,
  'magType': 'md',
  'place': '31km SE of Bodie, CA',
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 55,
  'net': 'nc',
  'nst': 26,
  'dmin': 0.2693,
  'rms': 0.09,
  'gap': 90,
  'magError': 0.249,
  'depthError': 4.22,
  'magNst': 20.0,
  'locationSource': 'nc',
  'm

In [ ]:
df = pd.DataFrame(all_records)

In [ ]:
df

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,pr2021001027,1609459299400,1611999858313,17.995100,-67.052500,5.00,1.750000,md,"2 km NNW of La Parguera, Puerto Rico",reviewed,...,120.0,0.120000,0.83,4.0,pr,pr,",origin,phase-data,",",pr2021001027,",",pr,",earthquake
1,nc73505360,1609459450210,1609797306182,38.047333,-118.728500,5.25,1.890000,md,"31km SE of Bodie, CA",reviewed,...,90.0,0.249000,4.22,20.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505360,",",nc,",earthquake
2,pr2021001028,1609459939760,1611999881681,17.976000,-66.842000,13.00,1.890000,md,"3 km SW of Indios, Puerto Rico",reviewed,...,208.0,0.080000,0.44,5.0,pr,pr,",origin,phase-data,",",pr2021001028,",",pr,",earthquake
3,pr2021001029,1609460047580,1611999899873,19.063100,-66.855500,8.00,3.140000,md,"63 km N of Hatillo, Puerto Rico",reviewed,...,284.0,0.040000,5.38,11.0,pr,pr,",origin,phase-data,",",pr2021001029,",",pr,",earthquake
4,pr2021001030,1609460184950,1612080776411,17.967600,-66.872000,19.00,1.660000,md,"2 km ESE of Maria Antonia, Puerto Rico",reviewed,...,303.0,0.170000,1.59,4.0,pr,pr,",origin,phase-data,",",pr2021001030,",",pr,",earthquake
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841079,ci41545856,1789210943710,1789211159810,33.930333,-118.309000,21.11,1.670000,ml,"4 km ESE of Lennox, CA",automatic,...,44.0,0.131892,0.36,31.0,ci,ci,",nearby-cities,origin,phase-data,scitech-link,",",ci41545856,",",ci,",earthquake
841080,ci41545864,1789211005610,1789211213691,33.237167,-118.066500,9.74,2.620000,ml,"27 km ESE of Avalon, CA",automatic,...,119.0,0.136963,1.05,27.0,ci,ci,",nearby-cities,origin,phase-data,scitech-link,",",ci41545864,",",ci,",earthquake
841081,nc75434212,1789211177390,1789211273872,38.829498,-122.814003,1.38,0.670000,md,"8 km NW of The Geysers, CA",automatic,...,125.0,0.310000,1.07,8.0,nc,nc,",nearby-cities,origin,phase-data,",",nc75434212,",",nc,",earthquake
841082,ci41545872,1789211519360,1789211653025,33.263000,-118.071999,14.26,2.973076,ml,"25 km ESE of Avalon, CA",automatic,...,114.0,0.226736,0.73,27.0,ci,ci,",nearby-cities,origin,phase-data,scitech-link,",",ci41545872,",",ci,",earthquake


In [ ]:
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,pr2021001027,1609459299400,1611999858313,17.995100,-67.0525,5.00,1.75,md,"2 km NNW of La Parguera, Puerto Rico",reviewed,...,120.0,0.120,0.83,4.0,pr,pr,",origin,phase-data,",",pr2021001027,",",pr,",earthquake
1,nc73505360,1609459450210,1609797306182,38.047333,-118.7285,5.25,1.89,md,"31km SE of Bodie, CA",reviewed,...,90.0,0.249,4.22,20.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505360,",",nc,",earthquake
2,pr2021001028,1609459939760,1611999881681,17.976000,-66.8420,13.00,1.89,md,"3 km SW of Indios, Puerto Rico",reviewed,...,208.0,0.080,0.44,5.0,pr,pr,",origin,phase-data,",",pr2021001028,",",pr,",earthquake
3,pr2021001029,1609460047580,1611999899873,19.063100,-66.8555,8.00,3.14,md,"63 km N of Hatillo, Puerto Rico",reviewed,...,284.0,0.040,5.38,11.0,pr,pr,",origin,phase-data,",",pr2021001029,",",pr,",earthquake
4,pr2021001030,1609460184950,1612080776411,17.967600,-66.8720,19.00,1.66,md,"2 km ESE of Maria Antonia, Puerto Rico",reviewed,...,303.0,0.170,1.59,4.0,pr,pr,",origin,phase-data,",",pr2021001030,",",pr,",earthquake


In [ ]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin',
       'rms', 'gap', 'magError', 'depthError', 'magNst', 'locationSource',
       'magSource', 'types', 'ids', 'sources', 'type'],
      dtype='str')

In [ ]:
df.columns.to_list()

['id',
 'time',
 'updated',
 'latitude',
 'longitude',
 'depth_km',
 'mag',
 'magType',
 'place',
 'status',
 'tsunami',
 'sig',
 'net',
 'nst',
 'dmin',
 'rms',
 'gap',
 'magError',
 'depthError',
 'magNst',
 'locationSource',
 'magSource',
 'types',
 'ids',
 'sources',
 'type']

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,tsunami,sig,net,nst,dmin,rms,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,pr2021001027,1609459299400,1611999858313,17.995100,-67.0525,5.00,1.75,md,"2 km NNW of La Parguera, Puerto Rico",reviewed,0,47,pr,4.0,0.0270,0.08,120.0,0.120,0.83,4.0,pr,pr,",origin,phase-data,",",pr2021001027,",",pr,",earthquake
1,nc73505360,1609459450210,1609797306182,38.047333,-118.7285,5.25,1.89,md,"31km SE of Bodie, CA",reviewed,0,55,nc,26.0,0.2693,0.09,90.0,0.249,4.22,20.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505360,",",nc,",earthquake
2,pr2021001028,1609459939760,1611999881681,17.976000,-66.8420,13.00,1.89,md,"3 km SW of Indios, Puerto Rico",reviewed,0,55,pr,6.0,0.2023,0.11,208.0,0.080,0.44,5.0,pr,pr,",origin,phase-data,",",pr2021001028,",",pr,",earthquake
3,pr2021001029,1609460047580,1611999899873,19.063100,-66.8555,8.00,3.14,md,"63 km N of Hatillo, Puerto Rico",reviewed,0,152,pr,16.0,0.6482,0.52,284.0,0.040,5.38,11.0,pr,pr,",origin,phase-data,",",pr2021001029,",",pr,",earthquake
4,pr2021001030,1609460184950,1612080776411,17.967600,-66.8720,19.00,1.66,md,"2 km ESE of Maria Antonia, Puerto Rico",reviewed,0,42,pr,4.0,0.1722,0.10,303.0,0.170,1.59,4.0,pr,pr,",origin,phase-data,",",pr2021001030,",",pr,",earthquake


In [ ]:
df["time"]

0         1609459299400
1         1609459450210
2         1609459939760
3         1609460047580
4         1609460184950
              ...      
841079    1789210943710
841080    1789211005610
841081    1789211177390
841082    1789211519360
841083    1789211638640
Name: time, Length: 841084, dtype: int64

In [ ]:
df[["time","updated"]]

,time,updated
0,1609459299400,1611999858313
1,1609459450210,1609797306182
2,1609459939760,1611999881681
3,1609460047580,1611999899873
4,1609460184950,1612080776411
...,...,...
841079,1789210943710,1789211159810
841080,1789211005610,1789211213691
841081,1789211177390,1789211273872
841082,1789211519360,1789211653025


In [ ]:
df["time"] = pd.to_datetime(
    df["time"],
    unit="ms",
    utc=True
)

df["updated"] = pd.to_datetime(
    df["updated"],
    unit="ms",
    utc=True
)

df[["time","updated"]]

,time,updated
0,2021-01-01 00:01:39.400000+00:00,2021-01-30 09:44:18.313000+00:00
1,2021-01-01 00:04:10.210000+00:00,2021-01-04 21:55:06.182000+00:00
2,2021-01-01 00:12:19.760000+00:00,2021-01-30 09:44:41.681000+00:00
3,2021-01-01 00:14:07.580000+00:00,2021-01-30 09:44:59.873000+00:00
4,2021-01-01 00:16:24.950000+00:00,2021-01-31 08:12:56.411000+00:00
...,...,...
841079,2026-09-12 11:02:23.710000+00:00,2026-09-12 11:05:59.810000+00:00
841080,2026-09-12 11:03:25.610000+00:00,2026-09-12 11:06:53.691000+00:00
841081,2026-09-12 11:06:17.390000+00:00,2026-09-12 11:07:53.872000+00:00
841082,2026-09-12 11:11:59.360000+00:00,2026-09-12 11:14:13.025000+00:00


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 841084 entries, 0 to 841083
Data columns (total 26 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   id              841084 non-null  str                
 1   time            841084 non-null  datetime64[ms, UTC]
 2   updated         841084 non-null  datetime64[ms, UTC]
 3   latitude        841084 non-null  float64            
 4   longitude       841084 non-null  float64            
 5   depth_km        841084 non-null  float64            
 6   mag             841017 non-null  float64            
 7   magType         841017 non-null  str                
 8   place           841084 non-null  str                
 9   status          841084 non-null  str                
 10  tsunami         841084 non-null  int64              
 11  sig             841084 non-null  int64              
 12  net             841084 non-null  str                
 13  nst             639411 no

In [ ]:
df.isnull()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,tsunami,sig,net,nst,dmin,rms,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841079,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
841080,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
841081,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
841082,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
df.isnull().sum()

id                     0
time                   0
updated                0
latitude               0
longitude              0
depth_km               0
mag                   67
magType               67
place                  0
status                 0
tsunami                0
sig                    0
net                    0
nst               201673
dmin              237014
rms                   44
gap               173802
magError          186103
depthError           351
magNst            175496
locationSource         0
magSource              0
types                  0
ids                    0
sources                0
type                   0
dtype: int64

In [ ]:
df["id"].duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

mag               67
magType           67
nst           201673
dmin          237014
rms               44
gap           173802
magError      186103
depthError       351
magNst        175496
dtype: int64

In [ ]:
null_summary = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_percentage": (df.isnull().mean() * 100).round(2)
})

null_summary[null_summary["null_count"] > 0]

,null_count,null_percentage
mag,67,0.01
magType,67,0.01
nst,201673,23.98
dmin,237014,28.18
rms,44,0.01
gap,173802,20.66
magError,186103,22.13
depthError,351,0.04
magNst,175496,20.87


In [ ]:
df = df.dropna(subset=["mag"])


In [ ]:
df["magType"].isnull().sum()

np.int64(0)

In [ ]:
df["rms"] = df["rms"].fillna(df["rms"].median())

In [ ]:
df["depthError"] = df["depthError"].fillna(
    df["depthError"].median()
)

In [ ]:
df["nst"].mean()


np.float64(23.82510538128866)

In [ ]:
df["gap"].median()


np.float64(100.0)

In [ ]:

df["magError"].describe()

count    654981.000000
mean          0.212803
std           0.317884
min           0.000000
25%           0.103000
50%           0.161000
75%           0.227735
max           6.190000
Name: magError, dtype: float64

In [ ]:
missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

missing_percentage[
    missing_percentage > 0
]

dmin        28.181476
nst         23.979539
magError    22.120361
magNst      20.859150
gap         20.665456
dtype: float64

In [ ]:
df.isnull().sum()[
    df.isnull().sum() > 0
]

nst         201672
dmin        237011
gap         173800
magError    186036
magNst      175429
dtype: int64

In [ ]:
df.to_csv("earthquake_data.csv", index=False)

In [ ]:
df = pd.read_csv("/Users/sreeharivenkataraman/Personal Files/DS_Projects/Project 1/Global Siesmic Trends/data/global_seismic_data_cleaned.csv")

NameError: name 'pd' is not defined

In [ ]:
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,tsunami,sig,net,nst,dmin,rms,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,pr2021001027,2021-01-01 00:01:39.400000+00:00,2021-01-30 09:44:18.313000+00:00,17.995100,-67.0525,5.00,1.75,md,"2 km NNW of La Parguera, Puerto Rico",reviewed,0,47,pr,4.0,0.0270,0.08,120.0,0.120,0.83,4.0,pr,pr,",origin,phase-data,",",pr2021001027,",",pr,",earthquake
1,nc73505360,2021-01-01 00:04:10.210000+00:00,2021-01-04 21:55:06.182000+00:00,38.047333,-118.7285,5.25,1.89,md,"31km SE of Bodie, CA",reviewed,0,55,nc,26.0,0.2693,0.09,90.0,0.249,4.22,20.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505360,",",nc,",earthquake
2,pr2021001028,2021-01-01 00:12:19.760000+00:00,2021-01-30 09:44:41.681000+00:00,17.976000,-66.8420,13.00,1.89,md,"3 km SW of Indios, Puerto Rico",reviewed,0,55,pr,6.0,0.2023,0.11,208.0,0.080,0.44,5.0,pr,pr,",origin,phase-data,",",pr2021001028,",",pr,",earthquake
3,pr2021001029,2021-01-01 00:14:07.580000+00:00,2021-01-30 09:44:59.873000+00:00,19.063100,-66.8555,8.00,3.14,md,"63 km N of Hatillo, Puerto Rico",reviewed,0,152,pr,16.0,0.6482,0.52,284.0,0.040,5.38,11.0,pr,pr,",origin,phase-data,",",pr2021001029,",",pr,",earthquake
4,pr2021001030,2021-01-01 00:16:24.950000+00:00,2021-01-31 08:12:56.411000+00:00,17.967600,-66.8720,19.00,1.66,md,"2 km ESE of Maria Antonia, Puerto Rico",reviewed,0,42,pr,4.0,0.1722,0.10,303.0,0.170,1.59,4.0,pr,pr,",origin,phase-data,",",pr2021001030,",",pr,",earthquake


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 841017 entries, 0 to 841016
Data columns (total 26 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              841017 non-null  str    
 1   time            841017 non-null  str    
 2   updated         841017 non-null  str    
 3   latitude        841017 non-null  float64
 4   longitude       841017 non-null  float64
 5   depth_km        841017 non-null  float64
 6   mag             841017 non-null  float64
 7   magType         841017 non-null  str    
 8   place           841017 non-null  str    
 9   status          841017 non-null  str    
 10  tsunami         841017 non-null  int64  
 11  sig             841017 non-null  int64  
 12  net             841017 non-null  str    
 13  nst             639345 non-null  float64
 14  dmin            604006 non-null  float64
 15  rms             841017 non-null  float64
 16  gap             667217 non-null  float64
 17  magError        65498

In [ ]:
df.describe()

,latitude,longitude,depth_km,mag,tsunami,sig,nst,dmin,rms,gap,magError,depthError,magNst
count,841017.000000,841017.000000,841017.000000,841017.000000,841017.000000,841017.000000,639345.000000,604006.000000,841017.000000,667217.000000,654981.000000,841017.000000,665588.000000
mean,39.306443,-110.205690,24.848130,1.664770,0.000791,68.801609,23.825105,0.693710,0.308944,117.206293,0.212803,1.950010,16.663794
std,20.480264,75.257863,55.757214,1.299263,0.028108,99.408416,22.835188,2.301097,0.280218,65.378601,0.317884,26.570471,27.528025
min,-84.493200,-179.999700,-10.000000,-5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000
25%,33.443167,-152.269500,3.970000,0.850000,0.000000,11.000000,10.000000,0.018960,0.100000,68.000000,0.103000,0.420000,6.000000
50%,38.816002,-122.756333,8.630000,1.400000,0.000000,30.000000,17.000000,0.065760,0.190000,100.000000,0.161000,0.760000,10.000000
75%,54.743667,-113.216000,18.900000,2.100000,0.000000,68.000000,30.000000,0.287000,0.500000,152.000000,0.227735,1.831000,18.000000
max,87.375200,179.999400,683.578000,8.800000,1.000000,2910.000000,619.000000,62.558000,8.550000,360.000000,6.190000,15982.200000,1027.000000


In [ ]:
%pip install sqlalchemy pymysql cryptography

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 3.9 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pymysql

In [ ]:
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "12345678"

connection = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)

cursor = connection.cursor()

cursor.execute(
    "CREATE DATABASE IF NOT EXISTS global_seismic_trends"
)

connection.commit()

cursor.close()
connection.close()

print("Database created successfully")

Database created successfully


In [ ]:
from sqlalchemy import create_engine

In [ ]:
DATABASE_NAME = "global_seismic_trends"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{DATABASE_NAME}"
)

In [ ]:
df[["time", "updated"]].head()


NameError: name 'df' is not defined

In [9]:
import pandas as pd

In [10]:
df = pd.read_csv("/Users/sreeharivenkataraman/Personal Files/DS_Projects/Project 1/Global Siesmic Trends/data/global_seismic_data_cleaned.csv")

In [11]:
df[["time", "updated"]].head()

,time,updated
0,2021-01-01 00:01:39.400000+00:00,2021-01-30 09:44:18.313000+00:00
1,2021-01-01 00:04:10.210000+00:00,2021-01-04 21:55:06.182000+00:00
2,2021-01-01 00:12:19.760000+00:00,2021-01-30 09:44:41.681000+00:00
3,2021-01-01 00:14:07.580000+00:00,2021-01-30 09:44:59.873000+00:00
4,2021-01-01 00:16:24.950000+00:00,2021-01-31 08:12:56.411000+00:00


In [12]:
df["time"] = df["time"].dt.tz_localize(None)
df["updated"] = df["updated"].dt.tz_localize(None)

AttributeError: Can only use .dt accessor with datetimelike values

In [13]:

df["time"] = pd.to_datetime(
    df["time"],
    errors="coerce",
    utc=True
)

df["updated"] = pd.to_datetime(
    df["updated"],
    errors="coerce",
    utc=True
)

In [14]:
df["time"] = df["time"].dt.tz_localize(None)
df["updated"] = df["updated"].dt.tz_localize(None)

In [15]:
df[["time", "updated"]].head()

,time,updated
0,2021-01-01 00:01:39.400,2021-01-30 09:44:18.313
1,2021-01-01 00:04:10.210,2021-01-04 21:55:06.182
2,2021-01-01 00:12:19.760,2021-01-30 09:44:41.681
3,2021-01-01 00:14:07.580,2021-01-30 09:44:59.873
4,2021-01-01 00:16:24.950,2021-01-31 08:12:56.411


In [16]:
df.to_sql(
    name="earthquakes",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Earthquake data inserted successfully")

Earthquake data inserted successfully


In [17]:
query = """
SELECT COUNT(*) AS total_earthquakes
FROM earthquakes
"""

result = pd.read_sql(query, engine)

result

,total_earthquakes
0,841017


In [18]:
query = """
SELECT *
FROM earthquakes
LIMIT 10
"""

test_df = pd.read_sql(query, engine)

test_df

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,pr2021001027,2021-01-01 00:01:39,2021-01-30 09:44:18,17.995100,-67.052500,5.00,1.75,md,"2 km NNW of La Parguera, Puerto Rico",reviewed,...,120.0,0.120,0.83,4.0,pr,pr,",origin,phase-data,",",pr2021001027,",",pr,",earthquake
1,nc73505360,2021-01-01 00:04:10,2021-01-04 21:55:06,38.047333,-118.728500,5.25,1.89,md,"31km SE of Bodie, CA",reviewed,...,90.0,0.249,4.22,20.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505360,",",nc,",earthquake
2,pr2021001028,2021-01-01 00:12:20,2021-01-30 09:44:42,17.976000,-66.842000,13.00,1.89,md,"3 km SW of Indios, Puerto Rico",reviewed,...,208.0,0.080,0.44,5.0,pr,pr,",origin,phase-data,",",pr2021001028,",",pr,",earthquake
3,pr2021001029,2021-01-01 00:14:08,2021-01-30 09:45:00,19.063100,-66.855500,8.00,3.14,md,"63 km N of Hatillo, Puerto Rico",reviewed,...,284.0,0.040,5.38,11.0,pr,pr,",origin,phase-data,",",pr2021001029,",",pr,",earthquake
4,pr2021001030,2021-01-01 00:16:25,2021-01-31 08:12:56,17.967600,-66.872000,19.00,1.66,md,"2 km ESE of Maria Antonia, Puerto Rico",reviewed,...,303.0,0.170,1.59,4.0,pr,pr,",origin,phase-data,",",pr2021001030,",",pr,",earthquake
5,pr2021001000,2021-01-01 00:18:18,2021-01-01 00:34:35,17.968300,-66.859800,13.00,2.49,md,"3 km ESE of Maria Antonia, Puerto Rico",reviewed,...,185.0,0.150,0.33,13.0,pr,pr,",origin,phase-data,",",pr2021001000,",",pr,",earthquake
6,nc73505365,2021-01-01 00:39:52,2021-01-04 21:50:06,37.472500,-118.851500,6.92,1.27,md,"18km WSW of Toms Place, CA",reviewed,...,131.0,0.184,1.41,19.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505365,",",nc,",earthquake
7,us6000d4p7,2021-01-01 00:40:36,2025-12-22 18:47:38,-52.290900,-4.762200,10.00,5.30,mww,southern Mid-Atlantic Ridge,reviewed,...,34.0,0.098,1.70,10.0,us,us,",losspager,origin,phase-data,shakemap,",",us6000d4p7,iscgem619630742,",",us,iscgem,",earthquake
8,hv72306177,2021-01-01 00:41:46,2021-01-01 00:44:54,19.177334,-155.489838,31.74,1.88,md,"3 km SSW of P?hala, Hawaii",automatic,...,92.0,0.540,1.11,5.0,hv,hv,",origin,phase-data,",",hv72306177,",",hv,",earthquake
9,pr2021001031,2021-01-01 00:46:46,2021-01-31 09:58:49,17.948000,-66.962300,7.00,1.64,md,"6 km WSW of Guánica, Puerto Rico",reviewed,...,260.0,0.090,0.27,3.0,pr,pr,",origin,phase-data,",",pr2021001031,",",pr,",earthquake


In [19]:
query = """
SELECT id, time, place, mag, depth_km
FROM earthquakes
ORDER BY mag DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,id,time,place,mag,depth_km
0,us6000qw60,2025-07-29 23:24:52,"2025 Kamchatka Peninsula, Russia Earthquake",8.8,35.000
1,ak0219neiszm,2021-07-29 06:15:49,"2021 Chignik, Alaska Earthquake",8.2,35.000
2,us6000f53e,2021-08-12 18:35:17,2021 South Sandwich Islands Earthquake,8.1,22.790
3,us7000dflf,2021-03-04 19:28:33,"2021 Kermadec Islands, New Zealand Earthquake",8.1,28.930
4,us6000tkt2,2026-08-14 21:58:22,"64 km NNW of Ende, Indonesia",7.8,10.000
5,us7000qx2g,2025-09-18 18:58:15,"140 km E of Petropavlovsk-Kamchatsky, Russia",7.8,27.000
6,us7000srb1,2026-06-07 23:37:42,"25 km SW of Kablalan, Philippines",7.8,56.991
7,us6000jllz,2023-02-06 01:17:34,"Pazarcik earthquake, Kahramanmaras earthquake ...",7.8,10.000
8,us6000kd0n,2023-05-19 02:57:03,southeast of the Loyalty Islands,7.7,18.053
9,us6000dg77,2021-02-10 13:19:56,southeast of the Loyalty Islands,7.7,10.000


In [20]:
query = """
SELECT AVG(mag) AS average_magnitude
FROM earthquakes;
"""

pd.read_sql(query, engine)

,average_magnitude
0,1.66477


In [21]:
query = """
SELECT
    YEAR(time) AS year,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY YEAR(time)
ORDER BY year;
"""

pd.read_sql(query, engine)

,year,earthquake_count
0,NaN,4863
1,2021.0,158626
2,2022.0,146043
3,2023.0,147129
4,2024.0,151214
5,2025.0,136415
6,2026.0,96727


In [22]:
query = """
SELECT *
FROM earthquakes
WHERE tsunami = 1
ORDER BY mag DESC;
"""

tsunami_df = pd.read_sql(query, engine)

tsunami_df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,us6000qw60,2025-07-29 23:24:52,2026-07-31 16:16:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,30.0,0.018,1.784,303.0,us,us,",dyfi,earthquake-name,event-sequence,finite-fa...",",at00t06p1k,us6000qw60,pt25210002,usauto6000qw60,",",at,us,pt,usauto,",earthquake
1,ak0219neiszm,2021-07-29 06:15:49,2026-04-15 18:42:11,55.3635,-157.8876,35.00,8.2,mww,"2021 Chignik, Alaska Earthquake",reviewed,...,NaN,NaN,0.000,NaN,ak,ak,",associate,dyfi,earthquake-name,finite-fault,g...",",ak0219neiszm,us6000f02w,ak0219nejcxv,at00qwzt...",",ak,us,ak,at,pt,usauto,iscgem,",earthquake
2,us7000dflf,2021-03-04 19:28:33,2025-12-22 18:58:47,-29.7228,-177.2794,28.93,8.1,mww,"2021 Kermadec Islands, New Zealand Earthquake",reviewed,...,21.0,0.034,3.300,81.0,us,us,",associate,dyfi,earthquake-name,finite-fault,g...",",at00qpgm3n,pt21063003,us7000dflf,usauto7000df...",",at,pt,us,usauto,iscgem,",earthquake
3,us7000qx2g,2025-09-18 18:58:15,2025-12-06 16:41:30,53.1426,160.7206,27.00,7.8,mww,"140 km E of Petropavlovsk-Kamchatsky, Russia",reviewed,...,76.0,0.024,1.880,173.0,us,us,",dyfi,finite-fault,general-text,ground-failure...",",at00t2ssp5,us7000qx2g,pt25261001,usauto7000qx2g,",",at,us,pt,usauto,",earthquake
4,us6000dg77,2021-02-10 13:19:56,2025-12-22 18:54:07,-23.0511,171.6566,10.00,7.7,mww,southeast of the Loyalty Islands,reviewed,...,15.0,0.042,1.800,54.0,us,us,",dyfi,finite-fault,general-text,ground-failure...",",us6000dg77,at00qobedd,pt21041004,iscgem619942...",",us,at,pt,iscgem,",earthquake


In [23]:
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,pr2021001027,2021-01-01 00:01:39.400,2021-01-30 09:44:18.313,17.995100,-67.0525,5.00,1.75,md,"2 km NNW of La Parguera, Puerto Rico",reviewed,...,120.0,0.120,0.83,4.0,pr,pr,",origin,phase-data,",",pr2021001027,",",pr,",earthquake
1,nc73505360,2021-01-01 00:04:10.210,2021-01-04 21:55:06.182,38.047333,-118.7285,5.25,1.89,md,"31km SE of Bodie, CA",reviewed,...,90.0,0.249,4.22,20.0,nc,nc,",nearby-cities,origin,phase-data,scitech-link,",",nc73505360,",",nc,",earthquake
2,pr2021001028,2021-01-01 00:12:19.760,2021-01-30 09:44:41.681,17.976000,-66.8420,13.00,1.89,md,"3 km SW of Indios, Puerto Rico",reviewed,...,208.0,0.080,0.44,5.0,pr,pr,",origin,phase-data,",",pr2021001028,",",pr,",earthquake
3,pr2021001029,2021-01-01 00:14:07.580,2021-01-30 09:44:59.873,19.063100,-66.8555,8.00,3.14,md,"63 km N of Hatillo, Puerto Rico",reviewed,...,284.0,0.040,5.38,11.0,pr,pr,",origin,phase-data,",",pr2021001029,",",pr,",earthquake
4,pr2021001030,2021-01-01 00:16:24.950,2021-01-31 08:12:56.411,17.967600,-66.8720,19.00,1.66,md,"2 km ESE of Maria Antonia, Puerto Rico",reviewed,...,303.0,0.170,1.59,4.0,pr,pr,",origin,phase-data,",",pr2021001030,",",pr,",earthquake


In [24]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day

In [25]:
df[
    ["time", "year", "month", "day"]
].head()

,time,year,month,day
0,2021-01-01 00:01:39.400,2021.0,1.0,1.0
1,2021-01-01 00:04:10.210,2021.0,1.0,1.0
2,2021-01-01 00:12:19.760,2021.0,1.0,1.0
3,2021-01-01 00:14:07.580,2021.0,1.0,1.0
4,2021-01-01 00:16:24.950,2021.0,1.0,1.0


In [26]:
df["day_of_week"] = df["time"].dt.day_name()

In [27]:
df[
    ["time", "day_of_week"]
].head()

,time,day_of_week
0,2021-01-01 00:01:39.400,Friday
1,2021-01-01 00:04:10.210,Friday
2,2021-01-01 00:12:19.760,Friday
3,2021-01-01 00:14:07.580,Friday
4,2021-01-01 00:16:24.950,Friday


In [28]:
def classify_depth(depth):

    if pd.isna(depth):
        return "Unknown"

    if depth < 70:
        return "Shallow"

    elif depth <= 300:
        return "Intermediate"

    else:
        return "Deep"
        

In [29]:
df["depth_category"] = (
    df["depth_km"]
    .apply(classify_depth)
)

In [30]:
df["depth_category"].value_counts()

depth_category
Shallow         759808
Intermediate     74264
Deep              6945
Name: count, dtype: int64

In [31]:
df["strong_flag"] = (
    df["mag"] >= 6
).astype(int)

In [32]:
df["strong_flag"].value_counts()

strong_flag
0    840243
1       774
Name: count, dtype: int64

In [33]:
df["destructive_flag"] = (
    df["mag"] >= 7
).astype(int)

In [34]:
df[
    [
        "mag",
        "strong_flag",
        "destructive_flag"
    ]
].head()

,mag,strong_flag,destructive_flag
0,1.75,0,0
1,1.89,0,0
2,1.89,0,0
3,3.14,0,0
4,1.66,0,0


In [39]:
df.columns.to_list()

['id',
 'time',
 'updated',
 'latitude',
 'longitude',
 'depth_km',
 'mag',
 'magType',
 'place',
 'status',
 'tsunami',
 'sig',
 'net',
 'nst',
 'dmin',
 'rms',
 'gap',
 'magError',
 'depthError',
 'magNst',
 'locationSource',
 'magSource',
 'types',
 'ids',
 'sources',
 'type',
 'year',
 'month',
 'day',
 'day_of_week',
 'depth_category',
 'strong_flag',
 'destructive_flag']

In [43]:
new_columns = [
    "country",
    "year",
    "month",
    "day",
    "day_of_week",
    "depth_category",
    "strong_flag",
    "destructive_flag"
]

df[["id"] + new_columns].head()

,id,country,year,month,day,day_of_week,depth_category,strong_flag,destructive_flag
0,pr2021001027,Puerto Rico,2021.0,1.0,1.0,Friday,Shallow,0,0
1,nc73505360,CA,2021.0,1.0,1.0,Friday,Shallow,0,0
2,pr2021001028,Puerto Rico,2021.0,1.0,1.0,Friday,Shallow,0,0
3,pr2021001029,Puerto Rico,2021.0,1.0,1.0,Friday,Shallow,0,0
4,pr2021001030,Puerto Rico,2021.0,1.0,1.0,Friday,Shallow,0,0


In [41]:
from sqlalchemy import text

In [42]:
df["country"] = (
    df["place"]
    .astype("string")
    .str.extract(r",\s*([^,]+)$", expand=False)
    .str.strip()
)

In [44]:
update_query = text("""
UPDATE earthquakes
SET
    country = :country,
    `year` = :year,
    `month` = :month,
    `day` = :day,
    day_of_week = :day_of_week,
    depth_category = :depth_category,
    strong_flag = :strong_flag,
    destructive_flag = :destructive_flag
WHERE id = :id
""")

In [45]:
update_df = df[
    [
        "id",
        "country",
        "year",
        "month",
        "day",
        "day_of_week",
        "depth_category",
        "strong_flag",
        "destructive_flag"
    ]
].copy()

In [46]:
update_df = update_df.astype(object).where(
    pd.notnull(update_df),
    None
)

In [47]:
records = update_df.to_dict(orient="records")

with engine.begin() as connection:
    connection.execute(
        update_query,
        records
    )

print("Derived columns updated successfully")

Derived columns updated successfully


In [48]:
print("DataFrame rows:", len(df))

DataFrame rows: 841017


In [49]:
df1 = pd.read_sql(
    "SELECT * FROM earthquakes",
    engine
)

In [50]:
df1

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,sources,type,country,year,month,day,day_of_week,depth_category,strong_flag,destructive_flag
0,ak02110dx5w9,2021-01-22 00:16:47,2021-02-04 01:02:38,51.282700,-176.369800,21.50,1.90,ml,"68 km SSE of Adak, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Shallow,0,0
1,ak02110dxt6y,2021-01-22 00:19:47,2021-02-04 20:55:39,59.873700,-141.534400,18.20,2.00,ml,"108 km WNW of Yakutat, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Shallow,0,0
2,ak02110dyvv9,2021-01-22 00:24:49,2021-02-04 20:55:40,52.213400,-168.376800,39.10,2.30,ml,"87 km SSE of Nikolski, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Shallow,0,0
3,ak02110dz981,2021-01-22 00:26:23,2021-02-04 01:02:51,59.944100,-152.177200,73.90,1.50,ml,"24 km W of Happy Valley, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Intermediate,0,0
4,ak02110e3pnu,2021-01-22 00:47:02,2021-04-13 17:05:49,55.003600,-157.637600,31.80,2.80,ml,Alaska Peninsula,reviewed,...,",us,ak,",earthquake,NaN,2021.0,1.0,22.0,Friday,Shallow,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841012,uw714099971,2026-09-11 09:15:24,2026-09-11 18:58:51,45.103333,-122.653500,34.06,1.21,ml,"6 km N of Scotts Mills, Oregon",reviewed,...,",uw,",earthquake,Oregon,2026.0,9.0,11.0,Friday,Shallow,0,0
841013,uw714100011,2026-09-11 12:42:06,2026-09-11 19:35:49,49.006000,-123.859000,42.59,1.77,ml,"3 km WNW of Ladysmith, Canada",reviewed,...,",uw,",earthquake,Canada,2026.0,9.0,11.0,Friday,Shallow,0,0
841014,uw714100021,2026-09-11 13:18:01,2026-09-11 18:29:42,46.724000,-121.135167,7.71,1.30,ml,"18 km SW of Nile, Washington",reviewed,...,",uw,",earthquake,Washington,2026.0,9.0,11.0,Friday,Shallow,0,0
841015,uw714100141,2026-09-11 18:31:18,2026-09-11 21:06:52,47.357333,-122.349167,20.35,1.09,md,"4 km NW of Federal Way, Washington",reviewed,...,",uw,",earthquake,Washington,2026.0,9.0,11.0,Friday,Shallow,0,0


In [51]:
df1.info

<bound method DataFrame.info of                   id                time             updated   latitude  \
0       ak02110dx5w9 2021-01-22 00:16:47 2021-02-04 01:02:38  51.282700   
1       ak02110dxt6y 2021-01-22 00:19:47 2021-02-04 20:55:39  59.873700   
2       ak02110dyvv9 2021-01-22 00:24:49 2021-02-04 20:55:40  52.213400   
3       ak02110dz981 2021-01-22 00:26:23 2021-02-04 01:02:51  59.944100   
4       ak02110e3pnu 2021-01-22 00:47:02 2021-04-13 17:05:49  55.003600   
...              ...                 ...                 ...        ...   
841012   uw714099971 2026-09-11 09:15:24 2026-09-11 18:58:51  45.103333   
841013   uw714100011 2026-09-11 12:42:06 2026-09-11 19:35:49  49.006000   
841014   uw714100021 2026-09-11 13:18:01 2026-09-11 18:29:42  46.724000   
841015   uw714100141 2026-09-11 18:31:18 2026-09-11 21:06:52  47.357333   
841016   uw714100441 2026-09-12 04:17:37 2026-09-12 05:39:00  48.421333   

         longitude  depth_km   mag magType  \
0      -176.369800   

In [52]:
df1.describe()

,time,updated,latitude,longitude,depth_km,mag,tsunami,sig,nst,dmin,rms,gap,magError,depthError,magNst,year,month,day,strong_flag,destructive_flag
count,836154,838662,841017.000000,841017.000000,841017.000000,841017.000000,841017.000000,841017.000000,639345.000000,604006.000000,841017.000000,667217.000000,654981.000000,841017.000000,665588.000000,836154.000000,836154.000000,836154.000000,841017.000000,841017.000000
mean,2023-10-11 01:15:16.267293,2023-11-21 06:52:09.241809,39.306443,-110.205690,24.848130,1.664770,0.000791,68.801609,23.825105,0.693710,0.308944,117.206293,0.212803,1.950010,16.663794,2023.300100,6.220422,15.638667,0.000920,0.000102
min,2021-01-01 00:01:39,2021-01-01 00:34:35,-84.493200,-179.999700,-10.000000,-5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,2021.000000,1.000000,1.000000,0.000000,0.000000
25%,2022-05-05 17:22:08.250000,2022-06-20 18:20:56,33.443167,-152.269500,3.970000,0.850000,0.000000,11.000000,10.000000,0.018960,0.100000,68.000000,0.103000,0.420000,6.000000,2022.000000,3.000000,8.000000,0.000000,0.000000
50%,2023-10-04 01:09:29,2023-12-14 20:57:18,38.816002,-122.756333,8.630000,1.400000,0.000000,30.000000,17.000000,0.065760,0.190000,100.000000,0.161000,0.760000,10.000000,2023.000000,6.000000,16.000000,0.000000,0.000000
75%,2025-03-04 19:54:01.250000,2025-04-16 19:25:09.500000,54.743667,-113.216000,18.900000,2.100000,0.000000,68.000000,30.000000,0.287000,0.500000,152.000000,0.227735,1.831000,18.000000,2025.000000,9.000000,23.000000,0.000000,0.000000
max,2026-09-12 11:13:59,2026-09-12 11:16:09,87.375200,179.999400,683.578000,8.800000,1.000000,2910.000000,619.000000,62.558000,8.550000,360.000000,6.190000,15982.200000,1027.000000,2026.000000,12.000000,31.000000,1.000000,1.000000
std,NaN,NaN,20.480264,75.257863,55.757214,1.299263,0.028108,99.408416,22.835188,2.301097,0.280218,65.378601,0.317884,26.570471,27.528025,1.648634,3.369162,8.811165,0.030323,0.010112


In [53]:
df1.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,sources,type,country,year,month,day,day_of_week,depth_category,strong_flag,destructive_flag
0,ak02110dx5w9,2021-01-22 00:16:47,2021-02-04 01:02:38,51.2827,-176.3698,21.5,1.9,ml,"68 km SSE of Adak, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Shallow,0,0
1,ak02110dxt6y,2021-01-22 00:19:47,2021-02-04 20:55:39,59.8737,-141.5344,18.2,2.0,ml,"108 km WNW of Yakutat, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Shallow,0,0
2,ak02110dyvv9,2021-01-22 00:24:49,2021-02-04 20:55:40,52.2134,-168.3768,39.1,2.3,ml,"87 km SSE of Nikolski, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Shallow,0,0
3,ak02110dz981,2021-01-22 00:26:23,2021-02-04 01:02:51,59.9441,-152.1772,73.9,1.5,ml,"24 km W of Happy Valley, Alaska",reviewed,...,",ak,",earthquake,Alaska,2021.0,1.0,22.0,Friday,Intermediate,0,0
4,ak02110e3pnu,2021-01-22 00:47:02,2021-04-13 17:05:49,55.0036,-157.6376,31.8,2.8,ml,Alaska Peninsula,reviewed,...,",us,ak,",earthquake,NaN,2021.0,1.0,22.0,Friday,Shallow,0,0


In [54]:
df1.sample()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,sources,type,country,year,month,day,day_of_week,depth_category,strong_flag,destructive_flag
349019,ci41253248,2025-08-05 05:14:22,2025-08-05 12:34:50,35.996167,-117.696667,3.24,0.69,ml,"20 km ENE of Little Lake, CA",reviewed,...,",ci,",earthquake,CA,2025.0,8.0,5.0,Tuesday,Shallow,0,0


In [55]:
df1.sample(10)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,sources,type,country,year,month,day,day_of_week,depth_category,strong_flag,destructive_flag
64550,ak022brnzttv,2022-09-13 17:49:52,2022-10-18 22:01:11,54.342900,-159.996800,23.5000,1.80,ml,"115 km SSE of Sand Point, Alaska",reviewed,...,",ak,",earthquake,Alaska,2022.0,9.0,13.0,Tuesday,Shallow,0,0
288814,ci39923407,2022-01-26 02:13:05,2022-01-26 15:35:32,33.755500,-116.675500,12.6700,0.58,ml,"4km ENE of Idyllwild, CA",reviewed,...,",ci,",earthquake,CA,2022.0,1.0,26.0,Wednesday,Shallow,0,0
369069,hv72758567,2021-10-16 11:47:19,2021-10-16 11:52:49,19.226334,-155.392838,31.5900,2.08,ml,"9 km ENE of P?hala, Hawaii",automatic,...,",hv,",earthquake,Hawaii,2021.0,10.0,16.0,Saturday,Shallow,0,0
745077,us7000g99p,2021-12-23 17:19:42,2022-03-01 16:39:56,-14.473900,-13.379500,10.0000,4.70,mb,southern Mid-Atlantic Ridge,reviewed,...,",us,",earthquake,NaN,2021.0,12.0,23.0,Thursday,Shallow,0,0
660878,tx2025npfvha,2025-07-12 10:15:23,2025-07-18 20:42:16,31.673000,-104.486000,2.7417,0.90,ml,"56 km S of Whites City, New Mexico",reviewed,...,",tx,",earthquake,New Mexico,2025.0,7.0,12.0,Saturday,Shallow,0,0
43501,ak0223b4jwht,2022-03-13 03:12:19,2022-04-14 16:02:38,51.728400,-176.766100,74.2000,2.10,ml,"18 km SSW of Adak, Alaska",reviewed,...,",ak,",earthquake,Alaska,2022.0,3.0,13.0,Sunday,Intermediate,0,0
581013,nn00893049,2025-02-08 04:15:03,2025-02-11 21:17:32,38.709200,-119.240600,6.1000,1.40,ml,"12 km SE of Smith Valley, Nevada",reviewed,...,",nn,",earthquake,Nevada,2025.0,2.0,8.0,Saturday,Shallow,0,0
592136,ok2022dxha,2022-02-25 10:17:15,2022-02-28 17:32:00,36.948667,-97.673667,6.1200,1.32,ml,"3 km NNW of Renfrow, Oklahoma",reviewed,...,",ok,",earthquake,Oklahoma,2022.0,2.0,25.0,Friday,Shallow,0,0
235838,av93038534,2025-04-09 02:30:28,2025-04-09 19:53:09,51.854500,-177.961000,3.6600,0.47,ml,"91 km W of Adak, Alaska",reviewed,...,",av,",earthquake,Alaska,2025.0,4.0,9.0,Wednesday,Shallow,0,0
388156,hv73714417,2024-01-11 06:39:02,2024-01-11 06:42:14,19.252501,-155.408340,32.9300,1.87,md,"9 km NE of Pāhala, Hawaii",automatic,...,",hv,",earthquake,Hawaii,2024.0,1.0,11.0,Thursday,Shallow,0,0
